<a href="https://colab.research.google.com/github/Vish204/RAG/blob/main/Document_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install PyMuPDF
!pip install sentence-transformers
!pip install faiss-cpu  # or faiss-gpu if you have GPU support
!pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.6 MB/s eta 0:00:00


In [3]:
import os
!pip install PyMuPDF # Added this line to ensure fitz is available
import fitz  # PyMuPDF
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

# Upload multiple files
uploaded_files = files.upload()

# Function to extract text from a single PDF file and split it into paragraphs
def extract_text_from_pdf(pdf_path):
    """Extracts text from a single PDF file and splits it into paragraphs."""
    text = ""
    with fitz.open(pdf_path) as pdf:
        for page in pdf:  # Loop through all pages
            text += page.get_text() + "\n"
    return text.strip().split('\n\n')  # Split into paragraphs

# Extract text from all uploaded PDFs
documents = []
for pdf_file in uploaded_files.keys():
    paragraphs = extract_text_from_pdf(pdf_file)  # Use the filename directly
    documents.append(paragraphs)

print(f"✅ Loaded {len(documents)} PDFs successfully!")
# print("📄 First 500 characters of the first document:\n", documents[0][0][:500])  # Shows first paragraph

# Load the pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model Loaded Successfully!")

# Generate embeddings for all paragraphs
doc_embeddings = []
paragraph_counts = []  # To keep track of the number of paragraphs in each document
for paragraphs in documents:
    embeddings = model.encode(paragraphs)
    doc_embeddings.append(embeddings)
    paragraph_counts.append(len(paragraphs))  # Store the count of paragraphs

doc_embeddings = np.vstack(doc_embeddings)  # Stack all embeddings
print("Embedding Shape:", doc_embeddings.shape)

# Reset FAISS Index
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # Create a new empty index
index.add(doc_embeddings)
print(f"✅ FAISS Index Rebuilt! Total Documents Indexed: {index.ntotal}")

def search_top_k_with_pdf(query, top_k=3):
    """Searches for the most relevant PDF content and displays related paragraphs."""
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    if len(indices[0]) == 0 or indices[0][0] == -1:
        return ["**No relevant documents found. Try a different query.**"]

    results = []
    for i, idx in enumerate(indices[0]):
        if idx >= 0:  # Ensure the index is valid
            # Find the corresponding document and paragraph index
            cumulative_count = 0
            for doc_index, count in enumerate(paragraph_counts):
                if idx < cumulative_count + count:
                    paragraph_index = idx - cumulative_count  # Calculate the paragraph index
                    paragraph = documents[doc_index][paragraph_index]  # Get the actual paragraph text
                    break
                cumulative_count += count

            # Highlight the query in the paragraph
            highlighted_paragraph = paragraph.replace(query, f"**{query}**")
            results.append(f"**Document {doc_index + 1}, Paragraph {paragraph_index + 1}:**\n{highlighted_paragraph}...\n")  # Display the paragraph
    return results

# Create an input box and search button
query_input = widgets.Text(placeholder="Enter your search query...")
search_button = widgets.Button(description="Search")
output_area = widgets.Output()

def on_search_click(b):
    query = query_input.value  # Get user input
    results = search_top_k_with_pdf(query, top_k=3)  # Get top 3 matches

    with output_area:
        output_area.clear_output()  # Clear previous results
        print("🔍 **Search Results:**\n")
        for i, result in enumerate(results, 1):
            print(f"**Top {i} Match:**\n{result}\n" + "-" * 50)  # Adding separator for clarity

# Link button click to search function
search_button.on_click(on_search_click)

# Display widgets
display(query_input, search_button, output_area)

Saving Dataset1.pdf to Dataset1.pdf
✅ Loaded 1 PDFs successfully!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Loaded Successfully!
Embedding Shape: (37, 384)
✅ FAISS Index Rebuilt! Total Documents Indexed: 37


Text(value='', placeholder='Enter your search query...')

Button(description='Search', style=ButtonStyle())

Output()